In [0]:
CREATE OR REPLACE TEMP VIEW bronze_orders_cleaned AS
SELECT
    UPPER(TRIM(order_id)) AS order_id,
    NULLIF(UPPER(TRIM(customer_id)), '') AS customer_id,
    NULLIF(UPPER(TRIM(store_id)), '') AS store_id,
    CASE
        WHEN UPPER(TRIM(order_status)) IN ('PLACED', 'DELIVERED', 'CANCELLED', 'CANCELED') THEN CASE WHEN UPPER(TRIM(order_status)) = 'CANCELED' THEN 'CANCELLED' ELSE UPPER(TRIM(order_status)) END
        ELSE NULL
    END AS order_status,
    -- Parse payment string as JSON and extract method
    CASE
        WHEN payment IS NOT NULL THEN UPPER(TRIM(get_json_object(payment, '$.method')))
        ELSE NULL
    END AS payment_method,
    CASE
        WHEN order_id IS NULL OR TRIM(channel) = '' THEN 'UNKNOWN'
        ELSE UPPER(TRIM(channel))
    END AS channel,
    CASE
        WHEN order_ts RLIKE '^\\d{4}-\\d{2}-\\d{2}(?:T| )\\d{2}:\\d{2}:\\d{2}$' THEN order_ts
        ELSE NULL
    END AS order_ts,
    CASE
        WHEN updated_at RLIKE '^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$' THEN updated_at
        ELSE NULL
    END AS updated_at,
    delivery,
    items,
    _rescued_data,
    _source_file_path,
    ingested_at
FROM fk_ce_2026_07_1372.bronze.orders
QUALIFY ROW_NUMBER() OVER (PARTITION BY UPPER(TRIM(order_id)) ORDER BY updated_at DESC) = 1;

In [0]:
-- Step 2: Quarantine rejected orders with C3 validation rules (fixed lookup paths)
CREATE OR REPLACE table fk_ce_2026_07_1372.silver.quarantine_orders AS
SELECT
  order_id,
  reject_reason,
  updated_at,
  to_json(named_struct(
    'order_id', order_id,
    'customer_id', customer_id,
    'store_id', store_id,
    'order_status', order_status,
    'payment_method', payment_method,
    'channel', channel,
    'order_ts', order_ts,
    'updated_at', updated_at,
    'delivery', delivery,
    'items', items,
    '_rescued_data', _rescued_data,
    '_source_file_path', _source_file_path,
    'ingested_at', ingested_at
  )) AS raw_record,
  current_timestamp() AS quarantined_at
FROM (
  SELECT *,
    CASE
      WHEN customer_id IS NULL THEN 'MISSING_CUSTOMER'
      WHEN NOT EXISTS (SELECT 1 FROM fk_ce_2026_07_1372.silver.customers c WHERE c.customer_id = bronze_orders_cleaned.customer_id) THEN 'UNKNOWN_CUSTOMER'
      WHEN store_id IS NULL THEN 'UNKNOWN_STORE'
      WHEN NOT EXISTS (SELECT 1 FROM fk_ce_2026_07_1372.silver.stores s WHERE s.store_id = bronze_orders_cleaned.store_id) THEN 'UNKNOWN_STORE'
      WHEN order_ts IS NULL THEN 'INVALID_TIMESTAMP'
      WHEN items IS NULL OR LENGTH(items) = 0 THEN 'EMPTY_ORDER'
      -- Unknown product, quantity, price checks to be expanded (nested items JSON parsing)
      ELSE NULL
    END AS reject_reason
  FROM bronze_orders_cleaned
) t
WHERE reject_reason IS NOT NULL;

In [0]:
-- Step 3: Extract valid orders for Silver and create Silver Orders
CREATE OR REPLACE TABLE fk_ce_2026_07_1372.silver.orders AS
SELECT
  order_id,
  customer_id,
  store_id,
  order_ts,
  CAST(date_format(CAST(order_ts AS DATE), 'yyyy-MM-dd') AS date) AS order_date,
  order_status,
  payment_method,
  channel,
  CAST(get_json_object(delivery, '$.promised_mins') AS INT) AS promised_mins,
  CAST(get_json_object(delivery, '$.actual_mins') AS INT) AS actual_mins,
  CAST(get_json_object(delivery, '$.delivery_fee') AS DECIMAL(10,2)) AS delivery_fee,
  updated_at
FROM bronze_orders_cleaned
WHERE order_id IS NOT NULL
  AND customer_id IS NOT NULL
  AND EXISTS (SELECT 1 FROM fk_ce_2026_07_1372.silver.customers c WHERE c.customer_id = bronze_orders_cleaned.customer_id)
  AND store_id IS NOT NULL
  AND EXISTS (SELECT 1 FROM fk_ce_2026_07_1372.silver.stores s WHERE s.store_id = bronze_orders_cleaned.store_id)
  AND order_ts IS NOT NULL
  AND items IS NOT NULL AND LENGTH(items) > 0;


-- Step 4: Extract items for valid orders and create Silver Order Items
CREATE OR REPLACE TABLE fk_ce_2026_07_1372.silver.order_items AS
SELECT
  o.order_id,
  CAST(item.line_number AS INT) AS line_number,
  item.product_id,
  CAST(item.quantity AS INT) AS quantity,
  CAST(item.unit_price AS DECIMAL(10,2)) AS unit_price,
  CAST(IF(item.discount_amount IS NULL, 0, item.discount_amount) AS DECIMAL(10,2)) AS discount_amount,
  (CAST(item.quantity AS INT) * CAST(item.unit_price AS DECIMAL(10,2))) AS gross_amount,
  (CAST(item.quantity AS INT) * CAST(item.unit_price AS DECIMAL(10,2)) - CAST(IF(item.discount_amount IS NULL, 0, item.discount_amount) AS DECIMAL(10,2))) AS net_amount
FROM silver_orders o
LATERAL VIEW explode(from_json(o.items, 'array<struct<line_number:int, product_id:string, quantity:int, unit_price:decimal(10,2), discount_amount:decimal(10,2)>>')) item_array AS item
WHERE item.product_id IS NOT NULL
  AND EXISTS (SELECT 1 FROM fk_ce_2026_07_1372.silver.products p WHERE p.product_id = item.product_id)
  AND item.quantity IS NOT NULL AND item.quantity > 0
  AND item.unit_price IS NOT NULL AND item.unit_price > 0;


In [0]:
-- Step 5: Incremental MERGE INTO for silver.orders
MERGE INTO fk_ce_2026_07_1372.silver.orders AS tgt
USING (
    SELECT
      order_id,
      customer_id,
      store_id,
      order_status,
      payment_method,
      channel,
      try_cast(order_ts AS TIMESTAMP) AS order_ts,
      try_cast(updated_at AS TIMESTAMP) AS updated_at,
      delivery,
      items,
      _rescued_data,
      _source_file_path,
      try_cast(ingested_at AS TIMESTAMP) AS ingested_at
    FROM fk_ce_2026_07_1372.silver.silver_orders
) AS src
ON tgt.order_id = src.order_id
WHEN MATCHED AND tgt.updated_at < src.updated_at THEN
    UPDATE SET
      customer_id = src.customer_id,
      store_id = src.store_id,
      order_status = src.order_status,
      payment_method = src.payment_method,
      channel = src.channel,
      order_ts = src.order_ts,
      updated_at = src.updated_at,
      delivery = src.delivery,
      items = src.items,
      _rescued_data = src._rescued_data,
      _source_file_path = src._source_file_path,
      ingested_at = src.ingested_at
WHEN NOT MATCHED THEN
    INSERT (order_id, customer_id, store_id, order_status, payment_method, channel, order_ts, updated_at, delivery, items, _rescued_data, _source_file_path, ingested_at)
    VALUES (src.order_id, src.customer_id, src.store_id, src.order_status, src.payment_method, src.channel, src.order_ts, src.updated_at, src.delivery, src.items, src._rescued_data, src._source_file_path, src.ingested_at);

-- Step 6: Incremental MERGE INTO for silver.order_items
MERGE INTO fk_ce_2026_07_1372.silver.order_items AS tgt
USING (
    SELECT
      order_id,
      CAST(line_number AS INT) AS line_number,
      product_id,
      CAST(quantity AS INT) AS quantity,
      CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
      CAST(discount_amount AS DECIMAL(10,2)) AS discount_amount,
      CAST(gross_amount AS DECIMAL(12,2)) AS gross_amount,
      CAST(net_amount AS DECIMAL(12,2)) AS net_amount
    FROM fk_ce_2026_07_1372.silver.order_items
) AS src
ON tgt.order_id = src.order_id AND tgt.line_number = src.line_number
WHEN MATCHED AND tgt.net_amount != src.net_amount THEN
    UPDATE SET
      product_id = src.product_id,
      quantity = src.quantity,
      unit_price = src.unit_price,
      discount_amount = src.discount_amount,
      gross_amount = src.gross_amount,
      net_amount = src.net_amount
WHEN NOT MATCHED THEN
    INSERT (order_id, line_number, product_id, quantity, unit_price, discount_amount, gross_amount, net_amount)
    VALUES (src.order_id, src.line_number, src.product_id, src.quantity, src.unit_price, src.discount_amount, src.gross_amount, src.net_amount);
